# Pyserini Demo on the MS MARCO Passage Dataset

This notebook replicates the BM25 baseline for the [MS MARCO passage ranking task](http://www.msmarco.org/) with [Pyserini](http://pyserini.io/), the Python interface to [Anserini](http://anserini.io).


## Installation


Install Python dependencies:


In [ ]:
!pip install pyserini==0.12.0
!pip install pytrec_eval
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

## Usage

You can use `search` function to search over an index. The topics (i.e., queries) are already distributed in Pyserini:

In [ ]:
from pyserini.search import get_topics

topics = get_topics('msmarco-passage-dev-subset')
print(f'{len(topics)} queries total')

6980 queries total


Let's take a look at a specific question. Topics often have different "fields": "title" is the one we want. (Again, this is just TREC parlance.)

In [ ]:
topics[1102400]['title']

'why do bears hibernate'

Next, we can initialize a searcher with a pre-built index, which Pyserini will automatically download:

In [ ]:
from pyserini.search import SimpleSearcher

searcher = SimpleSearcher.from_prebuilt_index('msmarco-passage')

Attempting to initialize pre-built index msmarco-passage.
/root/.cache/pyserini/indexes/index-msmarco-passage-20201117-f87c94.1efad4f1ae6a77e235042eff4be1612d already exists, skipping download.
Initializing msmarco-passage...


Now we can search:

In [ ]:
import json

hits = searcher.search(topics[1102400]['title'])

# Prints the first 10 hits
for i in range(0, 10):
    jsondoc = json.loads(hits[i].raw)
    print(f'{i+1:2} {hits[i].score:.5f} {jsondoc["contents"][:80]}...')

 1 17.33580 Why do Bears hibernate? March 31, 2010, Joan, Leave a comment. Why do bears hibe...
 2 13.23090 Why do bears hibernate? Watch this to discover how much effort is spent on survi...
 3 13.13570 Technically, as the other anwerer said, bears do not hibernate, but there isn't ...
 4 13.01460 It is a common misconception that bears hibernate during the winter. While bears...
 5 13.00390 To prepare for hibernation, grizzlies must prepare a den, and consume an immense...
 6 12.68940 Some zoo bears are fed year round, and do not hibernate. Since they do not under...
 7 12.55450 Bears in zoos will not hibernate if food is available, though they will slow dow...
 8 12.51710 All kinds of bears technically don't hibernate. They enter into a phase called t...
 9 12.43500 Date: 12-11-2012. It is a common misconception that bears hibernate during the w...
10 12.37460 While bears tend to slow down during the winter, they are not true hibernators. ...


The `hits` data structure holds the `docid`, the retrieval score, as well as the document content:

In [ ]:
from IPython.core.display import display, HTML
display(HTML('<div style="font-family: Times New Roman; padding-bottom:10px">' + hits[0].raw + '</div>'))

Let's run all the queries from the dev set:

In [ ]:
from pyserini.search import SimpleSearcher

def run_all_queries(file, topics, searcher):
    with open(file, 'w') as runfile:
        cnt = 0
        print('Running {} queries in total'.format(len(topics)))
        for id in topics:
            query = topics[id]['title']
            hits = searcher.search(query, 1000)
            for i in range(0, len(hits)):
                _ = runfile.write('{} Q0 {} {} {:.6f} Anserini\n'.format(id, hits[i].docid, i+1, hits[i].score))
            cnt += 1
            if cnt % 100 == 0:
                print(f'{cnt} queries completed')

searcher = SimpleSearcher.from_prebuilt_index('msmarco-passage')

run_all_queries('run-msmarco-passage-bm25.txt', topics, searcher)


Attempting to initialize pre-built index msmarco-passage.
/root/.cache/pyserini/indexes/index-msmarco-passage-20201117-f87c94.1efad4f1ae6a77e235042eff4be1612d already exists, skipping download.
Initializing msmarco-passage...
Running 6980 queries in total
100 queries completed
200 queries completed
300 queries completed
400 queries completed
500 queries completed
600 queries completed
700 queries completed
800 queries completed
900 queries completed
1000 queries completed
1100 queries completed
1200 queries completed
1300 queries completed
1400 queries completed
1500 queries completed
1600 queries completed
1700 queries completed
1800 queries completed
1900 queries completed
2000 queries completed
2100 queries completed
2200 queries completed
2300 queries completed
2400 queries completed
2500 queries completed
2600 queries completed
2700 queries completed
2800 queries completed
2900 queries completed
3000 queries completed
3100 queries completed
3200 queries completed
3300 queries comp

  Let's evaluate using `pytrec_eval`: the expected MAP, Recall@1000, NDCG@10 should be: X, Y, Z.

In [ ]:
!wget https://raw.githubusercontent.com/castorini/anserini-tools/28a938134b652a9153172edc0d82b7b765b66216/topics-and-qrels/qrels.msmarco-passage.dev-subset.txt

--2025-03-30 09:56:36--  https://raw.githubusercontent.com/castorini/anserini-tools/28a938134b652a9153172edc0d82b7b765b66216/topics-and-qrels/qrels.msmarco-passage.dev-subset.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 143300 (140K) [text/plain]
Saving to: ‘qrels.msmarco-passage.dev-subset.txt.1’

qrels.msmarco-passa 100%[===================>] 139.94K  --.-KB/s    in 0.1s    

2025-03-30 09:56:37 (1.17 MB/s) - ‘qrels.msmarco-passage.dev-subset.txt.1’ saved [143300/143300]



In [ ]:
import pytrec_eval
import numpy as np
with open('qrels.msmarco-passage.dev-subset.txt', 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)

with open('run-msmarco-passage-bm25.txt', 'r') as f_run:
    first_run = pytrec_eval.parse_run(f_run)

measures = {'map', 'ndcg_cut.10', 'recall.1000'}
evaluator = pytrec_eval.RelevanceEvaluator(qrel, measures)
results = evaluator.evaluate(first_run)
for measure in list(measures):
    mean_measure = np.mean([ele[measure.replace(".","_")] for ele in results.values()])
    print(measure, mean_measure)

recall.1000 0.8526146131805158
map 0.1925552058260252
ndcg_cut.10 0.22837308885666965


# Loading queries and document collection
## We provide two document and corpus loader functions to facilitate having access to the content of queries and documents by having their ids. This could help answer exercise two.

## downloading files

In [ ]:
!wget https://raw.githubusercontent.com/castorini/anserini-tools/28a938134b652a9153172edc0d82b7b765b66216/topics-and-qrels/topics.msmarco-passage.dev-subset.txt
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz

--2025-03-30 09:58:39--  https://raw.githubusercontent.com/castorini/anserini-tools/28a938134b652a9153172edc0d82b7b765b66216/topics-and-qrels/topics.msmarco-passage.dev-subset.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 290193 (283K) [text/plain]
Saving to: ‘topics.msmarco-passage.dev-subset.txt.1’

topics.msmarco-pass 100%[===================>] 283.39K  1.68MB/s    in 0.2s    

2025-03-30 09:58:39 (1.68 MB/s) - ‘topics.msmarco-passage.dev-subset.txt.1’ saved [290193/290193]

--2025-03-30 09:58:40--  https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 20.150.34.1
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|20.15

In [ ]:
!tar -xvzf collection.tar.gz

collection.tsv


In [ ]:
import tqdm
def read_collection(f_path):
  collection = {}
  with open(f_path, "r") as fp:
    for line in tqdm.tqdm(fp, desc="reading {}".format(f_path)):
      did, dtext = line.strip().split("\t")
      collection[did] = dtext
  return collection

In [ ]:
queries_dict = read_collection("topics.msmarco-passage.dev-subset.txt")

reading topics.msmarco-passage.dev-subset.txt: 6980it [00:00, 1666737.37it/s]


In [ ]:
documents_dict = read_collection("collection.tsv")

reading collection.tsv: 8841823it [00:14, 612037.84it/s]


#### Demo: representing the content of a document and a query

In [ ]:
#get id of first document in the dict.
print("id of first document in the documents_dict: ", list(documents_dict.keys())[0])
#get id of first query in the dict.
print("id of first query in the queries_dict: ", list(queries_dict.keys())[0])

print("\n----- here we print their content ----\n")

print("content of document id 0: ", documents_dict['0'])
print("content of query id 1048585: ", queries_dict['1048585'])

id of first document in the documents_dict:  0
id of first query in the queries_dict:  1048585

----- here we print their content ----

content of document id 0:  The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.
content of query id 1048585:  what is paula deen's brother


## Clues on answering questions manually


In [119]:
!wget https://www.dropbox.com/s/7wcskq2o9dr8qiv/bert-run.text

--2025-03-30 13:56:09--  https://www.dropbox.com/s/7wcskq2o9dr8qiv/bert-run.text
Resolving www.dropbox.com (www.dropbox.com)... 162.125.81.18, 2620:100:6031:18::a27d:5112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.81.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/70dcrbo1762d0n5i59vcf/bert-run.text?rlkey=o6f85anpcz3x0c6ozqrqrj1v4 [following]
--2025-03-30 13:56:09--  https://www.dropbox.com/scl/fi/70dcrbo1762d0n5i59vcf/bert-run.text?rlkey=o6f85anpcz3x0c6ozqrqrj1v4
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucf6e55c1cc7e208e8945384c34a.dl.dropboxusercontent.com/cd/0/inline/Cm04LxlRwNDhu-N_QRDNR1L3--hnbfEkPc0qKXZZt8fVvx9uDlwzJOoEJ1HGuCF1DI8gN_pJ4vbmicIFnkLJsulYJnuIZMp0SfPV9mVOHj_7ygOf23ASAfvFZerB4L5vY7ll9DsMviEpjQVKFDsqgsCv/file# [following]
--2025-03-30 13:56:10--  https://ucf6e55c1cc7e208e8945384c34a.dl.dropboxusercontent.com/cd/0/in

In [120]:
import pytrec_eval
#load qrel, bm25 ranking run file, and bert ranking run file
with open('qrels.msmarco-passage.dev-subset.txt', 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)

with open('run-msmarco-passage-bm25.txt', 'r') as f_run:
    bm25_run = pytrec_eval.parse_run(f_run)

with open('bert-run.text', 'r') as f_run:
    bert_run = pytrec_eval.parse_run(f_run)

# set measures and initialize evaluator
measures = {'recall.500', 'recall.10', 'map','ndcg_cut.10'}
evaluator = pytrec_eval.RelevanceEvaluator(qrel, measures)

# evaluate bm25 per query
bm25_results = evaluator.evaluate(bm25_run)

# evaluate bert per query
bert_results = evaluator.evaluate(bert_run)

In [121]:
bm25_recall500_zeros = {}
for query, measures_results in bm25_results.items():
  recall_500 = float(measures_results['recall_500'])
  if recall_500 == 0.0:
    bm25_recall500_zeros[query] = 0

#print last 10 queries with recall@500=0
print(list(bm25_recall500_zeros.keys())[-10:])

['1101552', '199572', '857943', '1083278', '320792', '717751', '329114', '1029791', '1083268', '1083267']


In [122]:
print(bert_results)

{'999942': {'map': 0.07692307692307693, 'recall_10': 0.0, 'ndcg_cut_10': 0.0}, '999921': {'map': 0.16666666666666666, 'recall_10': 1.0, 'ndcg_cut_10': 0.3562071871080222}, '999897': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999836': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999791': {'map': 0.3333333333333333, 'recall_10': 1.0, 'ndcg_cut_10': 0.5}, '999756': {'map': 0.14285714285714285, 'recall_10': 1.0, 'ndcg_cut_10': 0.3333333333333333}, '999691': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999685': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999641': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999637': {'map': 0.41666666666666663, 'recall_10': 1.0, 'ndcg_cut_10': 0.6052602440527057}, '999610': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999567': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999555': {'map': 1.0, 'recall_10': 1.0, 'ndcg_cut_10': 1.0}, '999552': {'map': 0.0, 'recall_10': 0.0, 'ndcg_cut_10': 0.0}, '999550

In [123]:
#print bert effectiveness for a query for that BM25 recall@500=0 and BERT recall@10=1
for query, measures_results in bert_results.items():
  recall_10 = float(measures_results['recall_10'])
  if recall_10 == 1.0 and query in bm25_recall500_zeros.keys():
    print("bert effectiveness for {} query in terms of recall@10 is:".format(query), bert_results[query]['recall_10'])
    break

bert effectiveness for 999921 query in terms of recall@10 is: 1.0


In [124]:
import numpy as np

bert_map = np.mean([one_query_result['map'] for one_query_result in bert_results.values()])
bert_ndcg_10 = np.mean([one_query_result['ndcg_cut_10'] for one_query_result in bert_results.values()])

print(f"BERT MAP results: {bert_map}")
print(f"BERT NDCG@10 results: {bert_ndcg_10}")


BERT MAP results: 0.3460835653874933
BERT NDCG@10 results: 0.39971943228978485


In [125]:
print(bm25_results)

{'1102330': {'map': 0.04, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '160885': {'map': 0.2, 'recall_10': 1.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.38685280723454163}, '1091115': {'map': 0.007518796992481203, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '613179': {'map': 0.5588235294117647, 'recall_10': 0.5, 'recall_500': 1.0, 'ndcg_cut_10': 0.6131471927654584}, '1091116': {'map': 0.007352941176470588, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '1092441': {'map': 0.08333333333333333, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '1091112': {'map': 0.5, 'recall_10': 1.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.6309297535714575}, '1093773': {'map': 0.00847457627118644, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '867490': {'map': 0.0049504950495049506, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '31595': {'map': 0.04, 'recall_10': 0.0, 'recall_500': 1.0, 'ndcg_cut_10': 0.0}, '54235': {'map': 0.003412969283276450

In [168]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 11 <= bm25_rank <= 500 and 1 <= bert_rank <= 10:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 1091115, Relevant Doc ID: 7941540, BM25 Rank: 133, BERT Rank: 3
Query detail: recurrent gastric cancer symptoms

Document detail: In more advanced stages of gastric cancer, the following signs and symptoms may occur: Blood in the stool. Vomiting. Weight loss for no known reason. Stomach pain. Jaundice (yellowing of eyes and skin). Ascites (build-up of fluid in the abdomen).

BM25 Top at top Doc ID: 2029617
BM25 Top Doc Content: Burning pain is common; food may worsen gastric ulcer symptoms but relieve duodenal ulcer symptoms. Acute complications include GI bleeding and perforation; chronic complications include gastric outlet obstruction, recurrence, and, when H. pylori infection is the cause, stomach cancer.

BERT Top at top Doc ID: 490054
BERT Top Doc Content: In more advanced stages of gastric cancer, the following signs and symptoms may occur: 1  Blood in the stool. 2  Vomiting. 3  Weight loss for no known reason. 4  Stomach pain. 5  Jaundice (yellowing of eyes and skin).

In [172]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 1 <= bm25_rank <= 10 and 11 <= bert_rank <= 1000:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 1087904, Relevant Doc ID: 7097707, BM25 Rank: 2, BERT Rank: 13
Query detail: what age do moles appear

Document detail: Most people do not develop new regular moles after the age of 30. Adults often develop non-mole growths like freckles, lentigines, liver spots, and seborrheic keratosis in later adulthood. New moles appearing after age 30 may require close observation, medical evaluation, and possible biopsy. A brand new mole in an adult may be a sign of an evolving abnormal mole or early melanoma. It is important to have any new or changing mole evaluated by your dermatologist.

BM25 Top at top Doc ID: 8726948
BM25 Top Doc Content: What are the causes and what do I watch for in moles, freckles, skin tags, lentigines and seborrheic keratoses? Most moles are benign. The only moles that are of medical concern are those that look different than other existing moles or those that first appear after age 20.

BERT Top at top Doc ID: 7212230
BERT Top Doc Content: Moles are growths 

In [173]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 501 <= bm25_rank <= 1000 and 1 <= bert_rank <= 10:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 995789, Relevant Doc ID: 7969429, BM25 Rank: 707, BERT Rank: 10
Query detail: where is the grafenberg spot located?

Document detail: The G-Spot is located from 1 to 3 inches up the anterior vaginal wall, between its opening and the urethra-it is a sensitive area that could be a part of the female prostate. There has been disagreement about whether the G-Spot exists at all as a separate structure.

BM25 Top at top Doc ID: 7969432
BM25 Top Doc Content: 1 Originally known as the Grafenberg Spot, the G-Spot was named after the gynecologist Ernst Gragr, fenberg who first described it in.  1944

BERT Top at top Doc ID: 2456408
BERT Top Doc Content: The Grafenberg spot, or G-spot, is an area located within the anterior (or front) wall of the vagina, about one centimetre from the surface and one-third to one-half way in from the vaginal opening (see illustration and text).he above view (A) shows the external view of the female vulva as normally seen when the woman is standing up. Vi